In [1]:
# Test data repair scripts for NY

import sys
import os
from dotenv import load_dotenv, find_dotenv
import matplotlib.pyplot as plt
import time
import pandas as pd
import numpy as np
import json

load_dotenv(find_dotenv())

ROOT_PATH = os.getenv("ROOT_PATH")
MY_DATA_PATH = os.getenv("MY_DATA_PATH")
RAW_DATA_PATH = os.getenv("RAW_DATA_PATH")
DEWEY_PATH = os.path.join(RAW_DATA_PATH, "dewey-downloads", "building-permits-united-states")

sys.path.append(os.path.join(ROOT_PATH, "scripts"))
import data_utils as du

sys.path.append(os.path.join(ROOT_PATH, "agent/scripts"))

from ny_data_repair import data_repair
MY_JURISDICTION = "New York"

INPUT_FILEPATH = os.path.join(MY_DATA_PATH, "processed_data", "permits_top50_sample.parquet")


In [2]:
df = pd.read_parquet(INPUT_FILEPATH)
sub_df = df[df["JURISDICTION"] == MY_JURISDICTION]

for col in ['FILE_DATE', 'PERMIT_DATE', 'FINAL_DATE']:
    sub_df[f'{col}_FLAG'] = ""

#sub_df_filled = sub_df.copy()
sub_df_filled = data_repair(sub_df)

assert(len(sub_df) == len(sub_df_filled))


In [3]:
print(f"FILE_DATE available (all): {sub_df['FILE_DATE'].notna().mean():.1%} -> {sub_df_filled['FILE_DATE'].notna().mean():.1%}")

print(f"PERMIT_DATE available (all): {sub_df['PERMIT_DATE'].notna().mean():.1%} -> {sub_df_filled['PERMIT_DATE'].notna().mean():.1%}")

print(f"FINAL_DATE available (all): {sub_df['FINAL_DATE'].notna().mean():.1%} -> {sub_df_filled['FINAL_DATE'].notna().mean():.1%}")

mask1 = sub_df['STATUS_NORMALIZED'].isin(['Active', 'Final'])
mask2 = sub_df_filled['STATUS_NORMALIZED'].isin(['Active', 'Final'])
print(f"PERMIT_DATE available (active/final): {sub_df.loc[mask1]['PERMIT_DATE'].notna().mean():.1%} -> {sub_df_filled.loc[mask2]['PERMIT_DATE'].notna().mean():.1%}")

mask1 = sub_df['STATUS_NORMALIZED'].isin(['Final'])
mask2 = sub_df_filled['STATUS_NORMALIZED'].isin(['Final'])
print(f"FINAL_DATE available (final): {sub_df.loc[mask1]['FINAL_DATE'].notna().mean():.1%} -> {sub_df_filled.loc[mask2]['FINAL_DATE'].notna().mean():.1%}")


FILE_DATE available (all): 87.1% -> 99.9%
PERMIT_DATE available (all): 63.1% -> 80.8%
FINAL_DATE available (all): 0.0% -> 43.9%
PERMIT_DATE available (active/final): 77.5% -> 87.6%
FINAL_DATE available (final): 0.0% -> 99.0%


In [4]:
print(sub_df['STATUS_NORMALIZED'].value_counts())
print(sub_df_filled['STATUS_NORMALIZED'].value_counts())

STATUS_NORMALIZED
Final        882
Active       569
In Review     86
Inactive      56
Name: count, dtype: int64
STATUS_NORMALIZED
Active       950
Final        887
In Review    104
Inactive      59
Name: count, dtype: int64


In [5]:
mask = sub_df_filled["PERMIT_DATE"].isna() 
#mask = sub_df_filled["JURISDICTION"].notna()
sample = sub_df_filled.loc[mask].sample(1).iloc[0]
DATA = sample["DATA"]
DATES_DATA = du.extract_date_fields(DATA) 

print(f"STATUS_NORMALIZED: {sample['STATUS_NORMALIZED']}    *Filled: {sample['STATUS_NORMALIZED_FLAG']}*")
print(f"RECORD_TYPE_ORIGINAL: {sample['RECORD_TYPE_ORIGINAL']}")
print(f"FILE_DATE: {sample['FILE_DATE']}       *Filled: {sample['FILE_DATE_FLAG']}*")
print(f"PERMIT_DATE: {sample['PERMIT_DATE']}   *Filled: {sample['PERMIT_DATE_FLAG']}*")
print(f"FINAL_DATE: {sample['FINAL_DATE']}     *Filled: {sample['FINAL_DATE_FLAG']}*")

print("DATES_DATA: ")
print(json.dumps(DATES_DATA, indent=2))



STATUS_NORMALIZED: In Review    *Filled: FILLED*
RECORD_TYPE_ORIGINAL: Alteration Type 1
FILE_DATE: 2015-02-02       *Filled: nan*
PERMIT_DATE: None   *Filled: nan*
FINAL_DATE: None     *Filled: nan*
DATES_DATA: 
{
  "filings": [
    {
      "paid": "02/02/2015",
      "assigned": "02/02/2015",
      "dobrundate": "06/04/2023 00:00:00",
      "fee_status": "EXEMPT",
      "fully_paid": "02/02/2015",
      "job_status": "F",
      "pre__filing_date": "02/02/2015",
      "job_status_descrp": "APPLICATION ASSIGNED TO PLAN EXAMINER",
      "latest_action_date": "02/02/2015",
      "special_action_date": "03/13/2015",
      "special_action_status": "W"
    }
  ]
}


In [6]:
print("DATA:")
print(json.dumps(json.loads(DATA), indent=2))



DATA:
{
  "filings": [
    {
      "lot": "00001",
      "paid": "02/02/2015",
      "bin__": "1001318",
      "block": "00096",
      "borough": "MANHATTAN",
      "cluster": "N",
      "gis_bin": "1001318",
      "house__": "11",
      "assigned": "02/02/2015",
      "job_type": "A1",
      "job_s1_no": "2459235",
      "city_owned": "Y",
      "dobrundate": "06/04/2023 00:00:00",
      "fee_status": "EXEMPT",
      "fully_paid": "02/02/2015",
      "job_status": "F",
      "landmarked": "Y",
      "loft_board": "N",
      "non_profit": "N",
      "owner_type": "OTHER GOV'T AGENCY",
      "street_name": "FULTON STREET",
      "gis_latitude": "40.706639",
      "gis_nta_name": "Battery Park City-Lower Manhattan",
      "initial_cost": "$0.00",
      "building_type": "OTHERS",
      "efiling_filed": "Y",
      "gis_longitude": "-74.003477",
      "building_class": "K6",
      "owner_sphone__": "2147417744",
      "total_est__fee": "$0.00",
      "existing_height": "0",
      "job_descr